# Site Dimension Data - Gold Layer

## Objective
Extract and deduplicate unique site attribute data from silver.silver_multimodal to build a standardized Site Dimension Gold Delta table (gold.gold_multimodal_dim_site).

## Data Flow
silver.silver_multimodal → Spark SQL / DataFrame → gold.gold_multimodal_dim_site

## Source
The underlying data comes from the multimodal transport network API.

## Input
Silver Delta table: silver.silver_multimodal

## Output
Gold Delta table: gold.gold_multimodal_dim_site

## Gold Layer Principle
The Gold layer delivers curated, dimensional models and business-level aggregations ready for reporting and analytics. This pipeline isolates unique geographical sites, coordinates, and labels to maintain a clean dimension table with strict entity integrity.

## Processing Steps
1. **Load Silver Data:** Read silver.silver_multimodal into PySpark.
2. **Extract Dimension:** Query distinct site IDs, labels, addresses, and geo-coordinates.
3. **Write to Gold:** Persist deduplicated dataset to gold.gold_multimodal_dim_site Delta table.

In [0]:
# Load data from silver schema
df_silver_multimodal=spark.table('workspace.silver.silver_multimodal')

In [0]:
# display the dataframe 
df_silver_multimodal.display()

In [0]:
%sql
SELECT * 
FROM workspace.silver.silver_multimodal 
limit 1000

## BUSINESS TRANSFORMATION AND MODELING

In [0]:
# Extract  Dimension Site Data
query_dim_site = """
SELECT DISTINCT
    site_id,
    site_label,
    site_label AS address,
    CAST(geo_coordinates.lat AS DOUBLE) AS latitude,
    CAST(geo_coordinates.lon AS DOUBLE) AS longitude
FROM workspace.silver.silver_multimodal
"""

df_dim_site = spark.sql(query_dim_site)

In [0]:
# Display df_dim_site 
df_dim_site.display()

# WRITING GOLD TABLE

In [0]:
df_dim_site\
    .write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("gold.gold_multimodal_dim_site")

# CHECKING THE GOLD TABLE

In [0]:
%sql
SELECT * 
FROM workspace.gold.gold_multimodal_dim_site 
LIMIT 10